# SVD — ConShift anchor / shift tables

Loads **`results/anchors_svd/{source}_to_{target}_shifts.tsv`** from `code/anchors_conshift.py --method svd` (same layout as Word2Vec: header row, `word`, `shift`, `count_*`).

*(Optional legacy: `results/svd_{a}_{b}.tsv` from `compute_distances.py` — two columns, no header — is also supported if present.)*

**Top-*N* tables:** one table per pair, columns **`word`**, **`shift`** (distance), **`count_a`**, **`count_b`**. Counts are **Word2Vec training token counts** (same proxy as the Word2Vec notebook), not SVD-specific.

**Pairs:** `arxiv`↔`yelp`, `arxiv`↔`ciao`, `yelp`↔`ciao` if the corresponding files exist.

In [13]:
from pathlib import Path
import pandas as pd

TOP_N = 10
CORPORA = ["arxiv", "yelp", "ciao", "reddit"]

candidates_repo = [
    Path(".").resolve(),
    Path("..").resolve(),
    Path("../..").resolve(),
]
REPO_ROOT = next(
    (p for p in candidates_repo if (p / "results").is_dir() and (p / "embeddings" / "word2vec").is_dir()),
    Path("..").resolve(),
)
RESULTS_ROOT = REPO_ROOT / "results"
ANCHORS_DIR = RESULTS_ROOT / "anchors_svd"

shift_files = []
for path in sorted(ANCHORS_DIR.glob("*_shifts.tsv")):
    pair_slug = path.stem.replace("_shifts", "")
    a, b = pair_slug.split("_to_", 1)
    shift_files.append((path, a, b))

if not shift_files:
    for a, b in [("arxiv", "yelp"), ("arxiv", "ciao"), ("yelp", "ciao")]:
        p = RESULTS_ROOT / f"svd_{a}_{b}.tsv"
        if p.is_file():
            shift_files.append((p, a, b))

print(f"REPO_ROOT     = {REPO_ROOT}")
print(f"RESULTS_ROOT  = {RESULTS_ROOT}")
print(f"ANCHORS_DIR   = {ANCHORS_DIR}")
print("Loaded:", [t[0].name for t in shift_files])

REPO_ROOT     = C:\work_study\github\UofT\CSC2611\synchronic-semantic-variation
RESULTS_ROOT  = C:\work_study\github\UofT\CSC2611\synchronic-semantic-variation\results
ANCHORS_DIR   = C:\work_study\github\UofT\CSC2611\synchronic-semantic-variation\results\anchors_svd
Loaded: ['arxiv_to_ciao_shifts.tsv', 'arxiv_to_yelp_shifts.tsv', 'yelp_to_ciao_shifts.tsv']


In [14]:
def parse_pair(pair_name: str):
    a, b = pair_name.split("_to_", 1)
    return a, b


_wv_cache: dict = {}


def wv_word_count(wv, w: str) -> int:
    if hasattr(wv, "get_vecattr"):
        try:
            c = wv.get_vecattr(w, "count")
            if c is not None:
                return int(c)
        except (KeyError, ValueError, TypeError):
            pass
    vocab = getattr(wv, "vocab", None)
    if vocab is not None and w in vocab:
        return int(vocab[w].count)
    return 0


def get_wv(corpus: str):
    if corpus not in _wv_cache:
        from gensim.models import Word2Vec

        path = REPO_ROOT / "embeddings" / "word2vec" / f"{corpus}.model"
        if not path.is_file():
            raise FileNotFoundError(f"Missing Word2Vec model: {path}")
        m = Word2Vec.load(str(path))
        _wv_cache[corpus] = m.wv
    return _wv_cache[corpus]


def attach_pair_counts(df: pd.DataFrame, src: str, tgt: str) -> pd.DataFrame:
    cs, ct = f"count_{src}", f"count_{tgt}"
    wv_a, wv_b = get_wv(src), get_wv(tgt)
    out = df.copy()
    out[cs] = [wv_word_count(wv_a, w) for w in out["word"]]
    out[ct] = [wv_word_count(wv_b, w) for w in out["word"]]
    return out


def load_svd_tsv(path: Path, src: str, tgt: str) -> pd.DataFrame:
    with open(path, encoding="utf-8") as f:
        first = f.readline()
    if first.lower().startswith("word\t"):
        df = pd.read_csv(path, sep="\t")
    else:
        df = pd.read_csv(path, sep="\t", header=None, names=["word", "shift"])
    pair_name = f"{src}_to_{tgt}"
    df["pair"] = pair_name
    cs, ct = f"count_{src}", f"count_{tgt}"
    if cs in df.columns and ct in df.columns:
        return df
    return attach_pair_counts(df, src, tgt)


def pairwise_top_n(
    df: pd.DataFrame, pair_name: str, top_n: int, *, largest: bool
) -> pd.DataFrame:
    src, tgt = parse_pair(pair_name)
    ca, cb = f"count_{src}", f"count_{tgt}"
    part = df.nlargest(top_n, "shift") if largest else df.nsmallest(top_n, "shift")
    out = part[["word", "shift", ca, cb]].copy()
    out.insert(0, "rank", range(1, top_n + 1))
    return out.rename(columns={ca: "count_a", cb: "count_b"})

## Top *N* highest distance (most shifted) per pair

In [15]:
from IPython.display import Markdown, display

for path, src, tgt in shift_files:
    df = load_svd_tsv(path, src, tgt)
    pair_name = df["pair"].iloc[0]
    display(
        Markdown(
            f"### `{pair_name}` — **count_a** = `{src}` (source), **count_b** = `{tgt}` (target)"
        )
    )
    display(pairwise_top_n(df, pair_name, TOP_N, largest=True))

### `arxiv_to_ciao` — **count_a** = `arxiv` (source), **count_b** = `ciao` (target)

,rank,word,shift,count_a,count_b
1892,1,dame,1.132959,8,89
7183,2,stimuli,1.128787,16,6
3196,3,fresh,1.127452,19,10205
5518,4,polarity,1.116597,7,7
5338,5,pedigrees,1.107231,5,5
3130,6,forgery,1.103459,6,5
934,7,butterfly,1.102440,8,369
5642,8,premises,1.099762,7,135
7772,9,trapdoor,1.099473,6,9
7914,10,underlay,1.098481,11,7


### `arxiv_to_yelp` — **count_a** = `arxiv` (source), **count_b** = `yelp` (target)

,rank,word,shift,count_a,count_b
184,1,agent's,1.164845,36,6
1064,2,chow,1.136940,10,929
2826,3,fisher,1.112422,56,34
5522,4,readable,1.109166,22,7
7071,5,tower,1.105995,33,1040
6973,6,theta,1.100561,250,10
6246,7,shin,1.100056,8,41
588,8,backwards,1.099584,9,225
7423,9,vegas,1.098890,10,1128
7509,10,vulnerable,1.098884,67,21


### `yelp_to_ciao` — **count_a** = `yelp` (source), **count_b** = `ciao` (target)

,rank,word,shift,count_a,count_b
17281,1,mays,1.176027,32,40
17762,2,minette,1.155981,55,10
27980,3,tangential,1.147968,5,6
30692,4,wale,1.142645,5,5
2256,5,beauticians,1.136913,6,8
12539,6,guido,1.136688,16,55
17728,7,milner,1.132198,8,28
13411,8,holt,1.132110,18,49
31627,9,yeti,1.131532,7,23
17023,10,mariana,1.130734,15,5


## Top *N* lowest distance (most stable) per pair

In [16]:
from IPython.display import Markdown, display

for path, src, tgt in shift_files:
    df = load_svd_tsv(path, src, tgt)
    pair_name = df["pair"].iloc[0]
    display(
        Markdown(
            f"### `{pair_name}` — **count_a** = `{src}` (source), **count_b** = `{tgt}` (target)"
        )
    )
    display(pairwise_top_n(df, pair_name, TOP_N, largest=False))

### `arxiv_to_ciao` — **count_a** = `arxiv` (source), **count_b** = `ciao` (target)

,rank,word,shift,count_a,count_b
2181,1,difficult,0.222857,419,6902
3720,2,improve,0.224035,899,1313
4149,3,june,0.238192,19,924
4514,4,march,0.238924,14,1108
2137,5,devices,0.239368,446,720
414,6,april,0.251044,8,1159
3766,7,increase,0.254668,470,1051
3604,8,http,0.256839,110,3934
5735,9,processor,0.263576,180,479
8362,10,years,0.266850,472,27237


### `arxiv_to_yelp` — **count_a** = `arxiv` (source), **count_b** = `yelp` (target)

,rank,word,shift,count_a,count_b
1775,1,dans,0.173652,6,16
3334,2,http,0.207414,110,1879
6674,3,students,0.207725,172,2276
2029,4,difficult,0.223764,419,3931
3813,5,june,0.235883,19,661
6291,6,significantly,0.253749,718,586
368,7,april,0.266433,8,599
4161,8,march,0.269946,14,661
7156,9,trees,0.284616,470,985
3024,10,games,0.287373,982,3827


### `yelp_to_ciao` — **count_a** = `yelp` (source), **count_b** = `ciao` (target)

,rank,word,shift,count_a,count_b
29123,1,trees,0.100558,985,1206
23869,2,room,0.100812,38799,15759
7790,3,different,0.104833,26582,31165
18039,4,monday,0.112764,3676,650
28763,5,tomato,0.114201,9408,1531
23872,6,rooms,0.118144,8318,3334
27110,7,strawberry,0.123450,3253,1915
26245,8,spacious,0.123684,2324,522
31398,9,wooden,0.123804,1033,2194
7803,10,difficult,0.124473,3931,6902


## Words in all loaded pairwise vocabularies

Inner join on `word`, **mean_shift** = mean of per-pair distances; **count_*** from Word2Vec.

In [17]:
if len(shift_files) >= 2:
    merged = None
    for path, a, b in shift_files:
        df = load_svd_tsv(path, a, b)
        pname = f"{a}_to_{b}"
        sub = df.rename(columns={"shift": f"shift_{pname}"})[["word", f"shift_{pname}"]]
        merged = sub if merged is None else merged.merge(sub, on="word", how="inner")
    shift_cols = [c for c in merged.columns if c.startswith("shift_")]
    merged["mean_shift"] = merged[shift_cols].mean(axis=1)
    for c in CORPORA:
        wv = get_wv(c)
        merged[f"count_{c}"] = [wv_word_count(wv, w) for w in merged["word"]]
    count_cols = [f"count_{c}" for c in CORPORA]
    print(f"Words in intersection of {len(shift_files)} SVD pair files: {len(merged)}")
    show_cols = ["word", "mean_shift"] + shift_cols + count_cols
    display(merged.nlargest(TOP_N, "mean_shift")[show_cols])
    display(merged.nsmallest(TOP_N, "mean_shift")[show_cols])
else:
    print("Need at least two SVD TSV files for merge.")

Words in intersection of 3 SVD pair files: 7441


,word,mean_shift,shift_arxiv_to_ciao,shift_arxiv_to_yelp,shift_yelp_to_ciao,count_arxiv,count_yelp,count_ciao
6392,stimuli,1.061471,1.128787,1.047718,1.007909,16,7,6
2395,errant,1.056154,1.080776,1.074124,1.013563,5,24,19
5839,scala,1.033872,1.073176,1.049617,0.978824,7,5,9
215,alfa,1.030812,1.009574,1.037764,1.045099,10,36,15
4174,milner,1.024104,1.039562,0.900552,1.132198,17,8,28
5292,ramifications,1.021290,0.964737,0.999161,1.099971,6,5,12
3191,hoffman,1.020109,1.003712,1.079011,0.977605,5,17,281
1666,cuckoo,1.018859,1.030597,1.043840,0.982141,16,6,68
276,ancestral,1.018399,1.020426,1.007382,1.027388,6,6,23
6961,truthfulness,1.018029,1.094284,1.036115,0.923689,14,6,6


,word,mean_shift,shift_arxiv_to_ciao,shift_arxiv_to_yelp,shift_yelp_to_ciao,count_arxiv,count_yelp,count_ciao
1954,difficult,0.190365,0.222857,0.223764,0.124473,419,3931,6902
6459,students,0.219287,0.277256,0.207725,0.172881,172,2276,1173
3690,june,0.221714,0.238192,0.235883,0.191067,19,661,924
1944,different,0.225239,0.275218,0.295665,0.104833,3299,26582,31165
6922,trees,0.232726,0.313004,0.284616,0.100558,470,985,1206
4028,march,0.235574,0.238924,0.269946,0.197853,14,661,1108
3228,http,0.235609,0.256839,0.207414,0.242573,110,1879,3934
354,april,0.248251,0.251044,0.266433,0.227275,8,599,1159
3,able,0.250616,0.297002,0.304133,0.150712,689,14257,15466
7425,years,0.251417,0.266850,0.313125,0.174277,472,26122,27237


## Optional: summary stats per file

In [18]:
for path, src, tgt in shift_files:
    df = load_svd_tsv(path, src, tgt)
    print(f"{path.name}: n={len(df)}, shift min/median/max = {df['shift'].min():.4f} / {df['shift'].median():.4f} / {df['shift'].max():.4f}")

arxiv_to_ciao_shifts.tsv: n=8385, shift min/median/max = 0.2229 / 0.8687 / 1.1330
arxiv_to_yelp_shifts.tsv: n=7698, shift min/median/max = 0.1737 / 0.8838 / 1.1648
yelp_to_ciao_shifts.tsv: n=31768, shift min/median/max = 0.1006 / 0.7868 / 1.1760
